In [ ]:
import os
import json

# Replace these with your actual Kaggle username + key
kaggle_username = "shanaeo"
kaggle_key = "KGAT_d3d00c04ea7deaef3c988a0132bd4fd8"

# Create kaggle.json file
kaggle_dir = "/root/.kaggle"
os.makedirs(kaggle_dir, exist_ok=True)

with open(f"{kaggle_dir}/kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)

# Set permissions
os.chmod(f"{kaggle_dir}/kaggle.json", 600)

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store
!unzip -q ecommerce-behavior-data-from-multi-category-store.zip

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors
100% 4.29G/4.29G [01:47<00:00, 43.0MB/s]



In [ ]:
import pandas as pd
import numpy as np


df = pd.read_csv("2019-Oct.csv")

# convert time
df["event_time"] = pd.to_datetime(df["event_time"])

# drop missing values (simple version)
df = df.dropna(subset=["event_type", "product_id", "price", "user_id"])

In [ ]:
df["is_view"] = (df["event_type"] == "view").astype(int)
df["is_cart"] = (df["event_type"] == "cart").astype(int)
df["is_purchase"] = (df["event_type"] == "purchase").astype(int)

In [ ]:
user_df = df.groupby("user_id").agg({
    "is_view": "sum",
    "is_cart": "sum",
    "is_purchase": "sum",
    "price": ["mean", "max"]
})

In [ ]:
user_df.columns = [
    "total_views",
    "total_carts",
    "total_purchases",
    "avg_price",
    "max_price"
]

user_df = user_df.reset_index()

In [ ]:
user_df["purchased"] = (user_df["total_purchases"] > 0).astype(int)

In [ ]:
# total number of events (overall activity level)
user_df["total_events"] = (
    user_df["total_views"] +
    user_df["total_carts"] +
    user_df["total_purchases"]
)

# fraction of views that lead to cart (intent signal)
user_df["cart_rate"] = user_df["total_carts"] / (user_df["total_views"] + 1)

# fraction of views that lead to purchase
user_df["purchase_rate"] = user_df["total_purchases"] / (user_df["total_views"] + 1)

# -----------------------------
# 3. TIME-BASED FEATURES
# -----------------------------

# extract hour of day and day of week
df["hour"] = df["event_time"].dt.hour
df["day"] = df["event_time"].dt.dayofweek

time_df = df.groupby("user_id").agg({
    "hour": ["mean", "std"],   # average browsing time + variability
    "day": "nunique"           # number of distinct active days
}).reset_index()

time_df.columns = ["user_id", "avg_hour", "hour_std", "active_days"]

# merge into user_df
user_df = user_df.merge(time_df, on="user_id", how="left")

# -----------------------------
# 4. PRODUCT / CATEGORY DIVERSITY
# -----------------------------

# number of unique categories user interacted with
cat_df = df.groupby("user_id")["category_id"].nunique().reset_index()
cat_df.columns = ["user_id", "num_categories"]

# number of unique products user interacted with
prod_df = df.groupby("user_id")["product_id"].nunique().reset_index()
prod_df.columns = ["user_id", "num_products"]

# merge both
user_df = user_df.merge(cat_df, on="user_id", how="left")
user_df = user_df.merge(prod_df, on="user_id", how="left")

# -----------------------------
# 5. ADDITIONAL PRICE FEATURES
# -----------------------------

price_df = df.groupby("user_id")["price"].agg([
    "std",   # variation in prices viewed
    "min"    # cheapest item viewed
]).reset_index()

price_df.columns = ["user_id", "price_std", "min_price"]

user_df = user_df.merge(price_df, on="user_id", how="left")

# -----------------------------
# 6. SIMPLE BINARY FLAGS (nice extra features)
# -----------------------------

# whether user ever added to cart
user_df["has_carted"] = (user_df["total_carts"] > 0).astype(int)

# whether user is high activity (arbitrary threshold)
user_df["high_activity"] = (user_df["total_views"] > 10).astype(int)

# whether user browses multiple categories
user_df["multi_category"] = (user_df["num_categories"] > 1).astype(int)

user_df = user_df.fillna(0)

# -----------------------------
# FINAL CHECK
# -----------------------------
print("Final shape:", user_df.shape)
user_df.head()

Final shape: (3022290, 20)


,user_id,total_views,total_carts,total_purchases,avg_price,max_price,purchased,total_events,cart_rate,purchase_rate,avg_hour,hour_std,active_days,num_categories,num_products,price_std,min_price,has_carted,high_activity,multi_category
0,33869381,1,0,0,769.650000,769.65,0,1,0.0,0.0,20.0,0.0,1,1,1,0.000000,769.65,0,0,0
1,64078358,1,0,0,0.000000,0.00,0,1,0.0,0.0,0.0,0.0,1,1,1,0.000000,0.00,0,0,0
2,183503497,1,0,0,15.770000,15.77,0,1,0.0,0.0,21.0,0.0,1,1,1,0.000000,15.77,0,0,0
3,184265397,6,0,0,111.706667,143.89,0,6,0.0,0.0,17.0,0.0,2,2,3,28.675972,79.77,0,0,1
4,195082191,1,0,0,161.880000,161.88,0,1,0.0,0.0,3.0,0.0,1,1,1,0.000000,161.88,0,0,0


In [ ]:
user_df.to_csv("user_level_data.csv", index=False)

from google.colab import files
files.download("user_level_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>